## Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [4]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [5]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 651.41 GB
MemAvailable: 806.82 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.c

## 2. Generating the cmw.txt file

In [ ]:
exp_id = "09-03-1"

In [ ]:
def process_file(file_path):
    # Read the file and count the number of rows
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    print(f"Number of rows: {len(lines)}")
    
    processed_lines = []

    # Process each line
    for line_idx, line in enumerate(lines):
        if line_idx > 10:
            break
        # Remove the trailing backslash and newline characters
        line = line.strip().rstrip('\\')
        
        # Split the line into the incorrect word and the correct words
        if '->' in line:
            incorrect_word, correct_words = line.split('->')
            
            # Split the correct words by comma and strip whitespace
            correct_words_list = [word.strip() for word in correct_words.split(',')]
            
            # Generate the output lines
            for correct_word in correct_words_list:
                processed_lines.append(f"{correct_word} {incorrect_word}")
        
    
    # Print each processed line
    for processed_line in processed_lines:
        print(processed_line)

# Define the path to your text file
file_path = "/nfs/homedirs/daro/git/quantization-reliability/data/cmw.txt"

# Run the function
process_file(file_path)

Number of rows: 4312
abandoned abandonned
aberration aberation
abilities abilityes
abilities abilties
ability abilty
abandon abondon
about abbout
about abotu
about a abouta
about it aboutit
about the aboutthe


In [ ]:
def process_and_save_file(input_file_path, output_file_path):
    # Read the input file and count the number of rows
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
    
    print(f"Number of rows: {len(lines)}")
    
    processed_lines = []

    # Process each line
    for line in lines:
        # Remove the trailing backslash and newline characters
        line = line.strip().rstrip('\\')
        
        # Split the line into the incorrect word and the correct words
        if '->' in line:
            incorrect_word, correct_words = line.split('->')
            
            # Split the correct words by comma and strip whitespace
            correct_words_list = [word.strip() for word in correct_words.split(',')]
            
            # Generate the output lines
            for correct_word in correct_words_list:
                processed_lines.append(f"{correct_word}: {incorrect_word}")
    
    # Write the processed lines to the output file
    with open(output_file_path, 'w') as output_file:
        for processed_line in processed_lines:
            output_file.write(processed_line + '\n')
    
    print(f"Processed lines have been saved to {output_file_path}")

# Define the paths to your text files
input_file_path = 'data/cmw.txt'
output_file_path = 'data/cmw_v2.txt'

# Run the function
process_and_save_file(input_file_path, output_file_path)

Number of rows: 4312
Processed lines have been saved to data/cmw_v2.txt


In [ ]:
def load_file_to_dict(file_path):
    cmw_dict = {}

    # Read the file and populate the dictionary
    with open(file_path, 'r') as file:
        lines = file.readlines()
        
        for line in lines:
            # Split each line by space to get the correct and incorrect words
            correct_word, incorrect_word = line.strip().split(':')
            cmw_dict[correct_word.strip()] = incorrect_word.strip()

    return cmw_dict

# Define the path to your processed text file
processed_file_path = 'data/cmw_v2.txt'

# Load the file into a dictionary
cmw_dict = load_file_to_dict(processed_file_path)

# Print the dictionary to verify
print(cmw_dict)

{'abandoned': 'abondoned', 'aberration': 'aberation', 'abilities': 'abilties', 'ability': 'abilty', 'abandon': 'adbandon', 'about': 'boaut', 'about a': 'abouta', 'about it': 'aboutit', 'about the': 'aboutthe', 'absence': 'absense', 'abandoning': 'abondoning', 'abandons': 'abondons', 'aborigine': 'aborigene', 'accessories': 'accesories', 'accident': 'acident', 'abortifacient': 'abortificant', 'abbreviate': 'abreviate', 'abbreviated': 'abreviated', 'abbreviation': 'abreviation', 'arbitrary': 'arbitary', 'abseil': 'absail', 'abseiling': 'absailing', 'absolutely': 'absolutly', 'absorption': 'absorbtion', 'abundance': 'abudance', 'abundances': 'abundancies', 'abundant': 'abundunt', 'abuts': 'abutts', 'academy': 'accademy', 'academic': 'acedemic', 'accused': 'acused', 'acceleration': 'accelleration', 'accession': 'accension', 'ascension': 'accension', 'acceptance': 'acceptence', 'acceptable': 'acceptible', 'accessible': 'accessable', 'accidentally': 'accidently', 'acclimatization': 'acclimit

In [ ]:
from nltk.corpus import wordnet

synonyms = wordnet.synsets("good")

In [ ]:
synonyms = []
for syn in wordnet.synsets("is"):
     for l in syn.lemmas():
          synonyms.append(l.name())
               
print(set(synonyms))

{'represent', 'be', 'exist', 'comprise', 'embody', 'follow', 'live', 'make_up', 'cost', 'equal', 'constitute', 'personify'}


## 3. Generating Typos

In [25]:
import random
import string
from nltk.corpus import wordnet
from emoji import EMOJI_DATA
from textblob import TextBlob
from googletrans import Translator
import emoji
from collections import defaultdict

translator = Translator()

def load_file_to_dict(file_path):
    cmw_dict = {}
    with open(file_path, 'r') as file:
        for line in file:
            correct_word, incorrect_word = line.strip().split(':')
            cmw_dict[correct_word.strip()] = incorrect_word.strip()
    return cmw_dict

def random_phrase_translation(word_list, num_translations):
    """Translate random words or phrases to a random foreign language."""
    languages = ['es', 'fr', 'de', 'it', 'ru', 'zh-cn', 'ja']
    for _ in range(num_translations):
        if len(word_list) < 2:
            return word_list
        
        is_phrase = random.choice([True, False]) if len(word_list) > 2 else False
        
        try:
            if is_phrase:
                start_index = random.randint(0, len(word_list) - 2)
                phrase = ' '.join(word_list[start_index:start_index+2])
                lang = random.choice(languages)
                translated_phrase = translator.translate(phrase, dest=lang).text
                word_list[start_index:start_index+2] = translated_phrase.split()
            else:
                word_idx = random.randint(0, len(word_list) - 1)
                lang = random.choice(languages)
                translated_word = translator.translate(word_list[word_idx], dest=lang).text
                word_list[word_idx] = translated_word
        except Exception as e:
            print(f"Translation error: {e}")
    
    return word_list

def random_insertion(word, num_insertions):
    for _ in range(num_insertions):
        pos = random.randint(0, len(word))
        char_to_insert = random.choice(string.ascii_letters)
        word = word[:pos] + char_to_insert + word[pos:]
    return word

def random_deletion(word, num_deletions):
    for _ in range(num_deletions):
        if len(word) > 1:
            pos = random.randint(0, len(word) - 1)
            word = word[:pos] + word[pos+1:]
    return word

def random_replacement(word, num_replacements):
    keyboard_adjacency = {
        'a': 'qwszy', 'b': 'vghn', 'c': 'xdfv', 'd': 'ersfxc', 'e': 'rdsw',
        'f': 'rtgvc', 'g': 'tyhbvf', 'h': 'ujnbg', 'i': 'ujko', 'j': 'uikmnh',
        'k': 'iolmj', 'l': 'opk', 'm': 'njk', 'n': 'bhjm', 'o': 'iklp',
        'p': 'ol', 'q': 'was', 'r': 'edft', 's': 'wedxza', 't': 'rfgy', 
        'u': 'yhji', 'v': 'cfgb', 'w': 'qase', 'x': 'yzsc', 'y': 'tghuxas',
        'z': 'asxtghu', '1': '2q', '2': '13w', '3': '24e', '4': '35r', '5': '46t',
        '6': '57tz', '7': '68u', '8': '79i', '9': '80o', '0': '9p'
    }
    for _ in range(num_replacements):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            if word[pos].lower() in keyboard_adjacency:
                replacement_char = random.choice(keyboard_adjacency[word[pos].lower()])
                word = word[:pos] + replacement_char + word[pos+1:]
    return word

def random_repetition(word, num_repetitions):
    for _ in range(num_repetitions):
        pos = random.randint(0, len(word))
        word = word[:pos] + word[pos-1:pos] + word[pos:]
    return word

def random_swapping(word, num_swaps):
    for _ in range(num_swaps):
        if len(word) > 1:
            pos = random.randint(0, len(word) - 2)
            word = word[:pos] + word[pos+1] + word[pos] + word[pos+2:]
    return word

def apply_cmw(word_list, num_replacements, cmw_dict):
    misspellable_words = [word for word in word_list if word.lower() in cmw_dict]
    num_replacements = min(num_replacements, len(misspellable_words))
    for _ in range(num_replacements):
        if misspellable_words:
            word_to_misspell = random.choice(misspellable_words)
            index = word_list.index(word_to_misspell)
            word_list[index] = cmw_dict[word_to_misspell.lower()]
            misspellable_words.remove(word_to_misspell)
    return word_list

def random_letter_case(word, num_case_changes):
    for _ in range(num_case_changes):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            word = word[:pos] + word[pos].swapcase() + word[pos+1:]
    return word

def synonym_replacement(word_list, num_replacements):
    for _ in range(num_replacements):
        replaceable_words = [w for w in word_list if wordnet.synsets(w)]
        if replaceable_words:
            word = random.choice(replaceable_words)
            synonyms = []
            for syn in wordnet.synsets(word):
                for lemma in syn.lemmas():
                    if lemma.name() != word:
                        synonyms.append(lemma.name())
            if synonyms:
                replacement = random.choice(synonyms)
                index = word_list.index(word)
                word_list[index] = replacement.replace('_', ' ')
    return word_list

def add_noise_characters(word, num_noises):
    for _ in range(num_noises):
        pos = random.randint(0, len(word))
        noise_char = random.choice(string.punctuation + string.digits)
        word = word[:pos] + noise_char + word[pos:]
    return word

def add_taxonomy(query, taxonomy, num_confusions):
    for _ in range(num_confusions):
        if taxonomy:
            random_taxonomy = random.choice(taxonomy)
            query = f"{random_taxonomy}. {query}"
    return query

def repeat_key_words(word_list, num_repetitions):
    for _ in range(num_repetitions):
        if word_list:
            word_idx = random.randint(0, len(word_list) - 1)
            word_list.insert(word_idx, word_list[word_idx])
    return word_list

def random_char_substitution(word, num_substitutions):
    char_map = {'O': '0', 'I': '1', 'S': '5', 'E': '3', 'A': '@', 'B': '8'}
    for _ in range(num_substitutions):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            if word[pos].upper() in char_map:
                word = word[:pos] + char_map[word[pos].upper()] + word[pos+1:]
    return word

def internet_slang_insertion(word_list, num_insertions):
    internet_slang = [
        'lol', 'rofl', 'idk', 'tbh', 'imo', 'fyi', 'brb', 'afk', 'tl;dr',
        'wtf', 'omg', 'smh', 'yolo', 'fomo', 'irl', 'tfw', 'ftw', 'imho',
        'iirc', 'asap', 'thx', 'hmu', 'xoxo', 'fwiw', 'ftfy', 'ama', 'eli5'
    ]
    for _ in range(num_insertions):
        pos = random.randint(0, len(word_list))
        word_list.insert(pos, random.choice(internet_slang))
    return word_list

class TrieNode:
    def __init__(self):
        self.children = defaultdict(TrieNode)
        self.emojis = []

def build_trie(emoji_data, language='en'):
    root = TrieNode()
    for emoji, data in emoji_data.items():
        if language in data:
            word = data[language].strip(':').lower()
            node = root
            for char in word:
                node = node.children[char]
            node.emojis.append(emoji)
    return root

def find_matching_emojis(word, trie):
    matches = []
    word = word.lower()
    for i in range(len(word)):
        node = trie
        for j in range(i, len(word)):
            if word[j] not in node.children:
                break
            node = node.children[word[j]]
            matches.extend(node.emojis)
    return matches

def emoji_substitution(word_list, num_substitutions, emoji_trie):
    for _ in range(num_substitutions):
        substitutable_words = [(i, word) for i, word in enumerate(word_list) if find_matching_emojis(word, emoji_trie)]
        if not substitutable_words:
            break
        index, word = random.choice(substitutable_words)
        matching_emojis = find_matching_emojis(word, emoji_trie)
        if matching_emojis:
            word_list[index] = random.choice(matching_emojis)
    return word_list

def text_to_speech_mishearing(word_list, num_mishearings):
    def get_homophones(word):
        homophones = []
        for synset in wordnet.synsets(word):
            for lemma in synset.lemmas():
                if lemma.name() != word and lemma.name().lower() not in homophones:
                    homophones.append(lemma.name().lower())
        return homophones

    for _ in range(num_mishearings):
        replaceable_words = [w for w in word_list if get_homophones(w)]
        if replaceable_words:
            word = random.choice(replaceable_words)
            index = word_list.index(word)
            homophones = get_homophones(word)
            if homophones:
                word_list[index] = random.choice(homophones)
    return word_list

def autocorrect_gone_wrong(word_list, num_autocorrects):
    for _ in range(num_autocorrects):
        if word_list:
            index = random.randint(0, len(word_list) - 1)
            word = word_list[index]
            blob = TextBlob(word)
            corrections = blob.correct()
            if corrections != word:
                word_list[index] = str(corrections)
    return word_list

def remove_punctuation_capitalization(query):
    return ''.join(char.lower() for char in query if char.isalnum() or char.isspace())

def keyword_only_query(query):
    stop_words = set(['the', 'is', 'at', 'which', 'on', 'a', 'an', 'are', 'was', 'were', 'in', 'that', 'to', 'for', 'of', 'with', 'by'])
    return ' '.join([word for word in query.split() if word.lower() not in stop_words])

def apply_typo_modifications(query, typo_dict, taxonomy=[], emoji_trie=None):
    cmw_file_path = 'data/cmw_v2.txt'
    cmw_dict = load_file_to_dict(cmw_file_path)

    words = query.split()

    for mod_type, num_modifications in typo_dict.items():
        if mod_type in ['remove_punctuation', 'keyword_only']:
            continue  # These are handled separately at the end

        if num_modifications > 0:
            if mod_type == 'CMW':
                words = apply_cmw(words, num_modifications, cmw_dict)
            elif mod_type == 'synonym':
                words = synonym_replacement(words, num_modifications)
            elif mod_type == 'noise':
                words = [add_noise_characters(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'char_substitution':
                words = [random_char_substitution(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'insertion':
                words = [random_insertion(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'deletion':
                words = [random_deletion(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'replacement':
                words = [random_replacement(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'repetition':
                words = [random_repetition(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'swapping':
                words = [random_swapping(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'LCC':
                words = [random_letter_case(word, 1) for word in random.sample(words, min(num_modifications, len(words)))]
            elif mod_type == 'emoji':
                words = emoji_substitution(words, num_modifications, emoji_trie)
            elif mod_type == 'tts_mishearing':
                words = text_to_speech_mishearing(words, num_modifications)
            elif mod_type == 'autocorrect':
                words = autocorrect_gone_wrong(words, num_modifications)
            elif mod_type == 'internet_slang':
                words = internet_slang_insertion(words, num_modifications)
            elif mod_type == 'phrase_translation':
                words = random_phrase_translation(words, num_modifications)
            elif mod_type == 'confusion':
                query = add_taxonomy(query, taxonomy, num_modifications)
            elif mod_type == 'repeat':
                words = repeat_key_words(words, num_modifications)

    modified_query = ' '.join(words)

    if typo_dict.get('remove_punctuation', False):
        modified_query = remove_punctuation_capitalization(modified_query)
    
    if typo_dict.get('keyword_only', False):
        modified_query = keyword_only_query(modified_query)
    
    return modified_query

# Example usage:
if __name__ == "__main__":
    # Preprocessing (do this once when initializing your script)
    emoji_trie = build_trie(EMOJI_DATA)
    
    query = "What is the favorite color of the sky that can be seen from the moon or the sun cloud bright low high rain"
    taxonomy = ["Munich", "Frankfurt", "Berlin"]
    typo_dict = {
        "insertion": 0,
        "deletion": 0,
        "replacement": 0,
        "repetition": 0,
        "swapping": 0,
        "CMW": 0,
        "LCC": 0,
        "synonym": 0,
        "noise": 0,
        "confusion": 0,
        "repeat": 0,
        "char_substitution": 0,
        "emoji": 10,
        "tts_mishearing": 0,
        "autocorrect": 0,
        "internet_slang": 0,
        "phrase_translation": 0,
        "remove_punctuation": False,
        "keyword_only": False
    }

    print("Original and modified query:")
    print(query)
    modified_query = apply_typo_modifications(query, typo_dict, taxonomy, emoji_trie)
    print(modified_query)

Original and modified query:
What is the favorite color of the sky that can be seen from the moon or the sun cloud bright low high rain
What is the favorite color of the sky that can be seen 🕉️ the moon or the ☀️ ☁ bright low high rain


In [28]:
EMOJI_DATA['🥇']

{'en': ':1st_place_medal:',
 'status': 2,
 'E': 3,
 'de': ':goldmedaille:',
 'es': ':medalla_de_oro:',
 'fr': ':médaille_d’or:',
 'ja': ':金メダル:',
 'ko': ':금메달:',
 'pt': ':medalha_de_ouro:',
 'it': ':medaglia_d’oro:',
 'fa': ':مدال_طلا:',
 'id': ':medali_emas:',
 'zh': ':金牌:',
 'ru': ':золотая_медаль:',
 'tr': ':birincilik_madalyası:',
 'ar': ':ميدالية_مركز_أول:'}

### Test modifications

In [5]:
import random

def test_typo_modifications():
    # Example query
    query = "What is the capital of France?"
    
    # Taxonomy for confusion modification
    taxonomy = ["Paris", "London", "Berlin", "Rome", "Madrid"]
    
    # List of all modification types
    modification_types = [
        "insertion", "deletion", "replacement", "repetition", "swapping",
        "CMW", "LCC", "synonym", "noise", "confusion", "repeat",
        "char_substitution", "emoji", "tts_mishearing",
        "autocorrect", "internet_slang", "remove_punctuation", "keyword_only",
        "phrase_translation"
    ]

    print("Typo Modification Tester")
    print("========================")
    print(f"Original query: {query}\n")

    # Apply each modification type individually
    for mod_type in modification_types:
        print(f"Applying {mod_type}:")
        for intensity in range(1, 6):  # Test intensities 1 to 5
            typo_dict = {mod_type: intensity if mod_type not in ["remove_punctuation", "keyword_only"] else True}
            modified_query = apply_typo_modifications(query, typo_dict, taxonomy)
            print(f"  Intensity {intensity}: {modified_query}")
        print()

    # Apply all modifications together
    print("Applying all modifications:")
    all_mods_dict = {mod: 2 for mod in modification_types if mod not in ["remove_punctuation", "keyword_only"]}
    all_mods_dict.update({"remove_punctuation": True, "keyword_only": True})
    
    for _ in range(5):  # Generate 5 examples with all modifications
        modified_query = apply_typo_modifications(query, all_mods_dict, taxonomy)
        print(f"  {modified_query}")

if __name__ == "__main__":
    test_typo_modifications()

Typo Modification Tester
Original query: What is the capital of France?

Applying insertion:
  Intensity 1: What is the capital ofz France?
  Intensity 2: WYhat is the capital of Fraunce?
  Intensity 3: WhatZ ils the capital oZf France?
  Intensity 4: What is thTe cpapital oPf Franfce?
  Intensity 5: Whant isV the capWital oXf FrAance?

Applying deletion:
  Intensity 1: What is the capital of Fance?
  Intensity 2: Wat i the capital of France?
  Intensity 3: Wha is the caital o France?
  Intensity 4: hat is the apital f rance?
  Intensity 5: Wha is th caital o Franc?

Applying replacement:
  Intensity 1: What is the capital if France?
  Intensity 2: Whzt is the capktal of France?
  Intensity 3: Wbat js the capital of trance?
  Intensity 4: Whaf is the dapital pf grance?
  Intensity 5: Wbat is tne capigal lf Ftance?

Applying repetition:
  Intensity 1: What is the capitaal of France?
  Intensity 2: What is tthe capital of France?
  Intensity 3: What iss the ccapital of France?
  Intensit

In [ ]:
modifications_to_exclude = [
  "language_switch"
]

modifications_needing_improvement = [
  "remove_punctuation",
  "internet_slang",
  "keyword_only",
  "autocorrect",
  "tts_mishearing",
  "emoji",
]

## 4. Semantic Similarity

In [ ]:
from gensim.models import KeyedVectors
from nlpia.data.loaders import get_data, BIGDATA_PATH

wordvector_path = os.path.join(BIGDATA_PATH, 'GoogleNews-vectors-negative300.bin.gz')

# Load a pre-trained word2vec model (this is just an example, the path should be to your downloaded model)
word_vectors = KeyedVectors.load_word2vec_format(wordvector_path, binary=True)

# Function to find similar words
def find_similar_words(word, topn=10):
    try:
        similar_words = word_vectors.most_similar(positive=[word], topn=topn)
        return [word for word, similarity in similar_words]
    except KeyError:
        return []

# Example usage
related_words = find_similar_words('philosopher')
print(related_words)

ImportError: cannot import name 'Mapping' from 'collections' (/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/collections/__init__.py)

In [ ]:
from nltk.corpus import wordnet as wn
import nltk

# Download WordNet data
nltk.download('wordnet')

def find_related_nouns(word):
    related_nouns = set()
    for synset in wn.synsets(word, pos=wn.NOUN):
        # Traverse through the hyponyms (subordinate concepts) and hypernyms (superordinate concepts)
        for lemma in synset.lemmas():
            related_nouns.add(lemma.name())
        for hypernym in synset.hypernyms():
            for lemma in hypernym.lemmas():
                related_nouns.add(lemma.name())
    return related_nouns

# Example usage
related_words = find_related_nouns('philosopher')
print(related_words)

[nltk_data] Downloading package wordnet to
[nltk_data]     /nfs/homedirs/daro/nltk_data...


{'student', 'scholar', 'scholarly_person', 'soul', 'individual', 'bookman', 'philosopher', 'person', 'somebody', 'mortal', 'someone'}


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to find semantically similar words
def find_similar_phrases(word, context_word):
    # Encode word and context into embeddings
    word_embedding = model.encode(word)
    context_embedding = model.encode(context_word)

    # Compute cosine similarity
    similarity_score = util.pytorch_cos_sim(word_embedding, context_embedding).item()
    
    return similarity_score

# Example usage
similarity = find_similar_phrases('philosopher', 'philosophy')
print(similarity)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


0.8044580817222595


## 5. Answer Correctness Handling

In [ ]:
from fuzzywuzzy import fuzz

# Example 1: Misspelling
str1 = "philosophy"
str2 = "filosophy"
ratio = fuzz.ratio(str1, str2)
print(f"Similarity between '{str1}' and '{str2}': {ratio}%")

# Example 2: Word Order Change
str3 = "John Smith from New York"
str4 = "Smith, John - New York"
ratio = fuzz.token_sort_ratio(str3, str4)
print(f"Similarity between '{str3}' and '{str4}': {ratio}%")

# Bonus: Partial String Matching
str5 = "The quick brown fox jumps over the lazy dog"
str6 = "brown fox"
ratio = fuzz.partial_ratio(str5, str6)
print(f"Partial match of '{str6}' in '{str5}': {ratio}%")

Similarity between 'philosophy' and 'filosophy': 84%
Similarity between 'John Smith from New York' and 'Smith, John - New York': 88%
Partial match of 'brown fox' in 'The quick brown fox jumps over the lazy dog': 100%


## 6. Generate PDF

In [4]:
import random
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY

def summarize_modifications():
    modifications = [
        {
            "name": "Random Insertion",
            "description": "Randomly inserts characters into a word.",
            "implementation": "Selects a random position in the word and inserts a random letter.",
            "example": ("Where is the capital of Germany?", "Where is thea capital of Germany?")
        },
        {
            "name": "Random Deletion",
            "description": "Randomly deletes characters from a word.",
            "implementation": "Selects a random position in the word and removes the character at that position.",
            "example": ("Where is the capital of Germany?", "Where is th capital of Germany?")
        },
        {
            "name": "Random Replacement",
            "description": "Randomly replaces characters in a word with adjacent keys on the keyboard.",
            "implementation": "Selects a random character in the word and replaces it with a random adjacent key on the keyboard.",
            "example": ("Where is the capital of Germany?", "Where is the capitsl of Germany?")
        },
        {
            "name": "Random Repetition",
            "description": "Randomly repeats characters in a word.",
            "implementation": "Selects a random position in the word and duplicates the character at that position.",
            "example": ("Where is the capital of Germany?", "Where is the capittal of Germany?")
        },
        {
            "name": "Random Swapping",
            "description": "Randomly swaps adjacent characters in a word.",
            "implementation": "Selects a random position in the word and swaps the character with its adjacent character.",
            "example": ("Where is the capital of Germany?", "Where is hte capital of Germany?")
        },
        {
            "name": "Common Misspelling Words (CMW)",
            "description": "Replaces words with their common misspelled variants.",
            "implementation": "Checks if a word is in a predefined dictionary of common misspellings and replaces it if found.",
            "example": ("Where is the capital of Germany?", "Where is the capitol of Germany?")
        },
        {
            "name": "Random Letter Case Change",
            "description": "Randomly changes the case of letters in a word.",
            "implementation": "Selects a random letter in the word and changes its case (upper to lower or vice versa).",
            "example": ("Where is the capital of Germany?", "Where is the caPital of Germany?")
        },
        {
            "name": "Synonym Replacement",
            "description": "Replaces words with their synonyms.",
            "implementation": "Uses NLTK's WordNet to find synonyms for a randomly selected word and replaces it.",
            "example": ("Where is the capital of Germany?", "Where is the metropolis of Germany?")
        },
        {
            "name": "Add Noise Characters",
            "description": "Adds random noise characters into words.",
            "implementation": "Inserts random punctuation or digit characters into a randomly selected word.",
            "example": ("Where is the capital of Germany?", "Where is the cap1ital of Germany?")
        },
        {
            "name": "Add Taxonomy",
            "description": "Adds a confusing taxonomy choice at the start of the query.",
            "implementation": "Prepends a random item from a predefined taxonomy list to the query.",
            "example": ("Where is the capital of Germany?", "Munich. Where is the capital of Germany?")
        },
        {
            "name": "Repeat Key Words",
            "description": "Repeats key words in the questions.",
            "implementation": "Selects a random word in the query and duplicates it at a random position.",
            "example": ("Where is the capital of Germany?", "Where is the capital capital of Germany?")
        },
        {
            "name": "Language Switching",
            "description": "Inserts words from a foreign language.",
            "implementation": "Inserts a random foreign word from a predefined list into the query.",
            "example": ("Where is the capital of Germany?", "Where is the hola capital of Germany?")
        },
        {
            "name": "Random Character Substitution",
            "description": "Substitutes characters with visually similar numbers or symbols.",
            "implementation": "Replaces certain letters with visually similar numbers or symbols (e.g., 'O' with '0').",
            "example": ("Where is the capital of Germany?", "Where is the capit@l of Germany?")
        },
        {
            "name": "Random Foreign Word Insertion",
            "description": "Inserts random foreign words into the word list.",
            "implementation": "Inserts a random foreign word from a predefined list into the query.",
            "example": ("Where is the capital of Germany?", "Where is the bonjour capital of Germany?")
        },
        {
            "name": "Random Phrase Translation",
            "description": "Translates random words or phrases to a random foreign language and back.",
            "implementation": "Uses a translation API to translate a random word to a foreign language and back.",
            "example": ("Where is the capital of Germany?", "Where is the Hauptstadt of Germany?")
        },
        {
            "name": "Language-based Structural Transformation",
            "description": "Transforms sentence structure to reflect foreign language syntax.",
            "implementation": "Rearranges the word order based on the syntax of a randomly chosen foreign language.",
            "example": ("Where is the capital of Germany?", "Where Germany of the capital is?")
        },
        {
            "name": "Emoji Substitution",
            "description": "Replace words with related emojis or insert emojis into the query.",
            "implementation": "1. Create a dictionary mapping words to related emojis.\n2. Randomly select words in the query.\n3. Replace selected words with their emoji equivalents or insert emojis after them.",
            "example": ("What is the weather like today?", "What is the ☀️ like today? 🌤️")
        },
        {
            "name": "Text-to-Speech Mishearing Simulation",
            "description": "Modify words to simulate common speech recognition errors.",
            "implementation": "1. Create a list of common homophones and near-homophones.\n2. Scan the query for words that have homophones.\n3. Randomly replace words with their homophones.",
            "example": ("How to pair my Bluetooth device?", "How to pear my Bluetooth device?")
        },
        {
            "name": "Autocorrect Gone Wrong",
            "description": "Simulate autocorrect mistakes by replacing words with similarly spelled but incorrect words.",
            "implementation": "1. Create a dictionary of common autocorrect mistakes.\n2. Scan the query for words that match keys in the dictionary.\n3. Replace matching words with their incorrect 'autocorrected' versions.",
            "example": ("How to make duck confit?", "How to make duck confident?")
        },
        {
            "name": "Internet Slang Insertion",
            "description": "Insert or replace words with common internet slang and abbreviations.",
            "implementation": "1. Create a dictionary of internet slang terms and their meanings.\n2. Randomly select words in the query.\n3. Replace selected words with their slang equivalents or insert slang terms.",
            "example": ("What are the best movies to watch?", "What are the best movies to watch? IMHO TBH")
        },
        {
            "name": "Punctuation and Capitalization Removal",
            "description": "Remove punctuation and capitalization to mimic quick, informal typing.",
            "implementation": "1. Remove all punctuation marks from the query.\n2. Convert the entire query to lowercase.",
            "example": ("What's the capital of France? Is it Paris?", "whats the capital of france is it paris")
        },
        {
            "name": "Keyword-Only Query",
            "description": "Reduce the query to essential keywords, simulating users who type minimal search terms.",
            "implementation": "1. Identify stop words (common words like 'the', 'is', 'are').\n2. Remove stop words and retain only key terms.",
            "example": ("What are the symptoms of the common cold?", "symptoms common cold")
        },
        {
            "name": "Symbolic Substitution",
            "description": "Replace words or parts of words with symbols or emojis that visually resemble them.",
            "implementation": "1. Create a dictionary mapping letters or words to visually similar symbols or emojis.\n2. Scan the query for matches in the dictionary.\n3. Replace matches with their symbolic or emoji counterparts.",
            "example": ("How to solve for x in algebra?", "H♡w t♡ s♡lve f♡r ✖ in ∀lgebra? 🧮")
        }
    ]
    return modifications

def create_pdf(filename):
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=18)
    story = []
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY))

    title = Paragraph("Typo and Query Modification Strategies", styles['Title'])
    story.append(title)
    story.append(Spacer(1, 12))

    for mod in summarize_modifications():
        name = Paragraph(f"<b>{mod['name']}</b>", styles['Heading2'])
        story.append(name)
        story.append(Spacer(1, 6))

        description = Paragraph(f"<b>Description:</b> {mod['description']}", styles['Normal'])
        story.append(description)
        story.append(Spacer(1, 6))

        implementation = Paragraph(f"<b>Implementation:</b> {mod['implementation']}", styles['Normal'])
        story.append(implementation)
        story.append(Spacer(1, 6))

        example = Paragraph(f"<b>Example:</b>", styles['Normal'])
        story.append(example)
        story.append(Spacer(1, 6))

        original = Paragraph(f"Original: {mod['example'][0]}", styles['Normal'])
        story.append(original)
        story.append(Spacer(1, 6))

        modified = Paragraph(f"Modified: {mod['example'][1]}", styles['Normal'])
        story.append(modified)
        story.append(Spacer(1, 12))

    doc.build(story)

if __name__ == "__main__":
    create_pdf("results/reliability_eval/typo_modification_summary.pdf")
    print("PDF report generated: typo_modification_summary.pdf")

PDF report generated: typo_modification_summary.pdf
